# NOESIS-Σ 1.5B — legacy Kaggle pilot on 2× Tesla T4 (data parallel)

**Legacy path.** The production target is eight 16 GB CUDA GPUs with the
profile in [`core/config_1.5b_production_256k.json`](../core/config_1.5b_production_256k.json),
documented in [`docs/09_LONG_CONTEXT_TRAINING.md`](../docs/09_LONG_CONTEXT_TRAINING.md).
Kaggle TPU sessions are not supported by the current NCCL and FSDP trainer.

These cells use plain distributed data parallelism at 512-token sequences for a
short engineering pilot. They do not exercise the 262,144-token production
profile.

Run cells in order. Cell 3 is a 20-step smoke test; only run Cell 4 after it passes.

In [ ]:
!pip install -q datasets transformers bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)


In [ ]:
import os, shutil, glob
src = None
for c in glob.glob('/kaggle/input/**/core', recursive=True):
    if os.path.exists(os.path.join(os.path.dirname(c), 'scripts', 'pretrain_1.5b_production.py')):
        src = os.path.dirname(c); break
assert src, "Bundle not found: add the NOESIS-SIGMA Bundle dataset as Input"
os.chdir('/kaggle/working')
for item in ['core', 'scripts', 'checkpoints']:
    shutil.copytree(os.path.join(src, item), f'/kaggle/working/{item}', dirs_exist_ok=True)
    print('copied', item)
!ls /kaggle/working/scripts

In [ ]:
# 4. SMOKE TEST (≈ a few minutes): both GPUs, random tokens, no internet needed
!cd /kaggle/working && torchrun --nproc_per_node=2 scripts/pretrain_1.5b_production.py \
    --config core/config_1.5b.json --steps 20 --batch_size 2 --seq_len 512 --grad_accum 2 \
    --warmup 5 --save_interval 20 --sample_interval 1000 --dry_data --output_dir checkpoints/smoke

In [ ]:
# 5. REAL RUN from scratch on BOTH GPUs (tokens/step = 2 GPUs x 2 x 511 x 16 = 32,704)
!cd /kaggle/working && torchrun --nproc_per_node=2 scripts/pretrain_1.5b_production.py \
    --config core/config_1.5b.json --steps 6000 --batch_size 2 --seq_len 512 --grad_accum 16 \
    --lr 3e-4 --warmup 200 --save_interval 250 --sample_interval 500 \
    --output_dir checkpoints/noesis_sigma_1.5b

In [ ]:
# 6. RESUME after a session ends (same command + --resume; raise --steps)
!cd /kaggle/working && torchrun --nproc_per_node=2 scripts/pretrain_1.5b_production.py \
    --config core/config_1.5b.json --steps 60000 --batch_size 2 --seq_len 512 --grad_accum 16 \
    --lr 3e-4 --warmup 200 --save_interval 250 --sample_interval 500 \
    --resume checkpoints/noesis_sigma_1.5b/latest.pt --output_dir checkpoints/noesis_sigma_1.5b